### Goal

This note book tries to answer wheter the RSD is important for galaxy-galxy lensing and galaxy clustering with DES Y3 constraining power.



In [1]:
import numpy as np
import pytest
import pyccl as ccl
from pyccl import CCLWarning

from pyccl.pk2d import Pk2D
from pyccl.correlations import correlation,correlation_binned

In [2]:
#load covariance matrix, source and lens sample for DES Y3
cov_file    = './cocoa/Cocoa/projects/desy3_real/data/des_y3_cov_unblinded_final.txt'
source_file = './cocoa/Cocoa/projects/desy3_real/data/des_y3_source.nz'
lens_file   = './cocoa/Cocoa/projects/desy3_real/data/des_y3_lens.nz'

cov_raw = np.loadtxt(cov_file)
ncov = int(np.max(cov_raw[:,0]))+1
cov = np.zeros((ncov, ncov))
for i in range(len(cov_raw)):
    ii = int(cov_raw[i, 0])
    jj = int(cov_raw[i, 1])
    element = cov_raw[i,2]
    cov[ii,jj] = element
    cov[jj,ii] = element

srcs_nzs = np.loadtxt(source_file)
lens_nzs = np.loadtxt(lens_file)

z_srcs = srcs_nzs[:,0]
srcs_nz = srcs_nzs[:,1:]
z_lens = lens_nzs[:,0]
lens_nz = lens_nzs[:,1:]

nsrcs = srcs_nz.shape[1]
nlens = lens_nz.shape[1]
ntheta = int(ncov/(nsrcs*(nsrcs+1)+nsrcs*nlens+nlens))

starts = [0, int(nsrcs*(nsrcs+1)*ntheta), int((nsrcs*(nsrcs+1)+nsrcs*nlens)*ntheta), int((nsrcs*(nsrcs+1)+nsrcs*nlens+nlens)*ntheta)]
cov_gammat = cov[starts[1]:starts[2],:][:,starts[1]:starts[2]]
cov_wtheta = cov[starts[2]:starts[3],:][:,starts[2]:starts[3]]
invcov_gammat = np.linalg.pinv(cov_gammat)
invcov_wtheta = np.linalg.pinv(cov_wtheta)

COSMO = ccl.Cosmology(
    Omega_c=0.26507647072945384,
    Omega_b=0.0495,
    Omega_k=0,
    h=0.6732,
    w0=-1,
    wa=0,
    A_s=2.1/1e9,
    n_s=0.96605,
    m_nu=0.06,
    Neff=3.046,
    mass_split='single',
    transfer_function='boltzmann_camb',
    matter_power_spectrum='camb',
    extra_parameters = {"camb": {"halofit_version": "takahashi",
                                 'AccuracyBoost': 1.0,
                                 'kmax':15,
                                 'dark_energy_model': 'ppf',
                                 'accurate_massive_neutrino_transfer': False,
                                 'k_per_logint': 15,
                                 }}
    )
h=0.6732

### Limber NoRSD

In [3]:
# setup lens and source tracer
srcs = []
lens = []
gbias =  [1.7, 1.7, 1.7, 2.0, 2.0]

for i in range(nsrcs):
    srcs.append(ccl.WeakLensingTracer(COSMO, dndz=(z_srcs, srcs_nz[:,i]), has_shear=True, n_samples=400))
for i in range(nlens):
    lens.append(ccl.NumberCountsTracer(COSMO, dndz=(z_lens, lens_nz[:,i]), bias=(z_lens,np.ones_like(z_lens)*gbias[i]), has_rsd=False, n_samples=400))

#calculate the power spectrum for calculation of correlation function
ELLMAX_NOLIMBER = 150
logLMIN = np.log(ELLMAX_NOLIMBER)
logLMAX = np.log(60000+1)
NCell = 300
dlogL = (logLMAX - logLMIN)/(NCell - 1)

ells = np.zeros(ELLMAX_NOLIMBER+NCell)
for i in range(ELLMAX_NOLIMBER):
    ells[i] = i
for i in range(NCell):
    ells[i+ELLMAX_NOLIMBER] = np.exp(logLMIN + dlogL*i)
ells = ells[1:]

gammat_cl = []
wtheta_cl = []

for i in range(nlens):
    for j in range(nsrcs):
        gammat_cl.append(ccl.angular_cl(COSMO, lens[i], srcs[j], ells, l_limber=-1) )
        
for i in range(nlens):
    wtheta_cl.append(ccl.angular_cl(COSMO, lens[i], lens[i], ells, l_limber=-1))
    
#calculate the correlation function
tmin = 2.5
tmax = 250.
#ntheta = 15

logtmin = np.log(tmin)
logtmax = np.log(tmax)
logdt=(logtmax - logtmin)/ntheta
fac = (2./3.)
thetas = np.zeros(ntheta)

for i in range(ntheta):
    thetamin = np.exp(logtmin + (i + 0.)*logdt)
    thetamax = np.exp(logtmin + (i + 1.)*logdt)
    thetas[i] = fac * (thetamax**3 - thetamin**3) / (thetamax*thetamax    - thetamin*thetamin)
thetas /= 60

theta_b  = np.geomspace(tmin, tmax, ntheta+1) / 60
theta_u  = theta_b[1:]
theta_l  = theta_b[:-1]

ncombo_gammat = int(nsrcs*nlens)
ncombo_wtheta = int(nlens)

gammat, wtheta = [],[]

for i in range(ncombo_gammat):
    gammat.append(correlation_binned(COSMO, ell=ells, C_ell=gammat_cl[i], theta_min=theta_l, theta_max=theta_u, type='NG', method='Legendre'))
gammat = np.concatenate(gammat)

for i in range(ncombo_wtheta):
    wtheta.append(correlation_binned(COSMO, ell=ells, C_ell=wtheta_cl[i], theta_min=theta_l, theta_max=theta_u, type='NN', method='Legendre'))
wtheta = np.concatenate(wtheta)
 
limber_noRSD_gammat = gammat
limber_noRSD_wtheta = wtheta

### Limber RSD

In [4]:
# setup lens and source tracer
srcs = []
lens = []
nsrcs = srcs_nz.shape[1]
nlens = lens_nz.shape[1]
gbias =  [1.7, 1.7, 1.7, 2.0, 2.0]

for i in range(nsrcs):
    srcs.append(ccl.WeakLensingTracer(COSMO, dndz=(z_srcs, srcs_nz[:,i]), has_shear=True, n_samples=400))
for i in range(nlens):
    lens.append(ccl.NumberCountsTracer(COSMO, dndz=(z_lens, lens_nz[:,i]), bias=(z_lens,np.ones_like(z_lens)*gbias[i]), has_rsd=True, n_samples=400))

#calculate the power spectrum for calculation of correlation function
ELLMAX_NOLIMBER = 150
logLMIN = np.log(ELLMAX_NOLIMBER)
logLMAX = np.log(60000+1)
NCell = 300
dlogL = (logLMAX - logLMIN)/(NCell - 1)

ells = np.zeros(ELLMAX_NOLIMBER+NCell)
for i in range(ELLMAX_NOLIMBER):
    ells[i] = i
for i in range(NCell):
    ells[i+ELLMAX_NOLIMBER] = np.exp(logLMIN + dlogL*i)
ells = ells[1:]

gammat_cl = []
wtheta_cl = []

for i in range(nlens):
    for j in range(nsrcs):
        gammat_cl.append(ccl.angular_cl(COSMO, lens[i], srcs[j], ells, l_limber=-1) )
        
for i in range(nlens):
    wtheta_cl.append(ccl.angular_cl(COSMO, lens[i], lens[i], ells, l_limber=-1))
    
#calculate the correlation function
tmin = 2.5
tmax = 250.
#ntheta = 15

logtmin = np.log(tmin)
logtmax = np.log(tmax)
logdt=(logtmax - logtmin)/ntheta
fac = (2./3.)
thetas = np.zeros(ntheta)

for i in range(ntheta):
    thetamin = np.exp(logtmin + (i + 0.)*logdt)
    thetamax = np.exp(logtmin + (i + 1.)*logdt)
    thetas[i] = fac * (thetamax**3 - thetamin**3) / (thetamax*thetamax    - thetamin*thetamin)
thetas /= 60

theta_b  = np.geomspace(tmin, tmax, ntheta+1) / 60
theta_u  = theta_b[1:]
theta_l  = theta_b[:-1]

ncombo_gammat = int(nsrcs*nlens)
ncombo_wtheta = int(nlens)

gammat, wtheta = [],[]

for i in range(ncombo_gammat):
    gammat.append(correlation_binned(COSMO, ell=ells, C_ell=gammat_cl[i], theta_min=theta_l, theta_max=theta_u, type='NG', method='Legendre'))
gammat = np.concatenate(gammat)

for i in range(ncombo_wtheta):
    wtheta.append(correlation_binned(COSMO, ell=ells, C_ell=wtheta_cl[i], theta_min=theta_l, theta_max=theta_u, type='NN', method='Legendre'))
wtheta = np.concatenate(wtheta)
 
limber_RSD_gammat = gammat
limber_RSD_wtheta = wtheta

In [5]:
print('!under Limber!')
chi2_gammat = (limber_RSD_gammat - limber_noRSD_gammat)@invcov_gammat@(limber_RSD_gammat - limber_noRSD_gammat)
print(f'The RSD raises chi2 {chi2_gammat:.2f} for gammat')
chi2_wtheta = (limber_RSD_wtheta - limber_noRSD_wtheta)@invcov_wtheta@(limber_RSD_wtheta - limber_noRSD_wtheta)
print(f'The RSD raises chi2 {chi2_wtheta:.2f} for wtheta')

!under Limber!
The RSD raises chi2 0.00 for gammat
The RSD raises chi2 0.15 for wtheta


### noLimber noRSD

In [6]:
# setup lens and source tracer
srcs = []
lens = []
gbias =  [1.7, 1.7, 1.7, 2.0, 2.0]

for i in range(nsrcs):
    srcs.append(ccl.WeakLensingTracer(COSMO, dndz=(z_srcs, srcs_nz[:,i]), has_shear=True, n_samples=400))
for i in range(nlens):
    lens.append(ccl.NumberCountsTracer(COSMO, dndz=(z_lens, lens_nz[:,i]), bias=(z_lens,np.ones_like(z_lens)*gbias[i]), has_rsd=False, n_samples=400))

#calculate the power spectrum for calculation of correlation function
ELLMAX_NOLIMBER = 150
logLMIN = np.log(ELLMAX_NOLIMBER)
logLMAX = np.log(60000+1)
NCell = 300
dlogL = (logLMAX - logLMIN)/(NCell - 1)

ells = np.zeros(ELLMAX_NOLIMBER+NCell)
for i in range(ELLMAX_NOLIMBER):
    ells[i] = i
for i in range(NCell):
    ells[i+ELLMAX_NOLIMBER] = np.exp(logLMIN + dlogL*i)
ells = ells[1:]

gammat_cl = []
wtheta_cl = []

for i in range(nlens):
    for j in range(nsrcs):
        gammat_cl.append(ccl.angular_cl(COSMO, lens[i], srcs[j], ells, l_limber=ELLMAX_NOLIMBER) )
        
for i in range(nlens):
    wtheta_cl.append(ccl.angular_cl(COSMO, lens[i], lens[i], ells, l_limber=ELLMAX_NOLIMBER))
    
#calculate the correlation function
tmin = 2.5
tmax = 250.
#ntheta = 15

logtmin = np.log(tmin)
logtmax = np.log(tmax)
logdt=(logtmax - logtmin)/ntheta
fac = (2./3.)
thetas = np.zeros(ntheta)

for i in range(ntheta):
    thetamin = np.exp(logtmin + (i + 0.)*logdt)
    thetamax = np.exp(logtmin + (i + 1.)*logdt)
    thetas[i] = fac * (thetamax**3 - thetamin**3) / (thetamax*thetamax    - thetamin*thetamin)
thetas /= 60

theta_b  = np.geomspace(tmin, tmax, ntheta+1) / 60
theta_u  = theta_b[1:]
theta_l  = theta_b[:-1]

ncombo_gammat = int(nsrcs*nlens)
ncombo_wtheta = int(nlens)

gammat, wtheta = [],[]

for i in range(ncombo_gammat):
    gammat.append(correlation_binned(COSMO, ell=ells, C_ell=gammat_cl[i], theta_min=theta_l, theta_max=theta_u, type='NG', method='Legendre'))
gammat = np.concatenate(gammat)

for i in range(ncombo_wtheta):
    wtheta.append(correlation_binned(COSMO, ell=ells, C_ell=wtheta_cl[i], theta_min=theta_l, theta_max=theta_u, type='NN', method='Legendre'))
wtheta = np.concatenate(wtheta)
 
nolimber_noRSD_gammat = gammat
nolimber_noRSD_wtheta = wtheta

/project/chihway/junzhou/Code-comparison/ccl/pyccl/errors.py:22: CCLWarning: Nchi must be a positive integer. Setting to match tracer with large chi samples x 2.
  warnings_builtin.warn(*args, **kwargs)
/project/chihway/junzhou/Code-comparison/ccl/pyccl/errors.py:22: CCLWarning: chi_min must be greater than zero.Setting to default 1e-6 Mpc.
  warnings_builtin.warn(*args, **kwargs)


### noLimber RSD

In [7]:
# setup lens and source tracer
srcs = []
lens = []
nsrcs = srcs_nz.shape[1]
nlens = lens_nz.shape[1]
gbias =  [1.7, 1.7, 1.7, 2.0, 2.0]

for i in range(nsrcs):
    srcs.append(ccl.WeakLensingTracer(COSMO, dndz=(z_srcs, srcs_nz[:,i]), has_shear=True, n_samples=400))
for i in range(nlens):
    lens.append(ccl.NumberCountsTracer(COSMO, dndz=(z_lens, lens_nz[:,i]), bias=(z_lens,np.ones_like(z_lens)*gbias[i]), has_rsd=True, n_samples=400))

#calculate the power spectrum for calculation of correlation function
ELLMAX_NOLIMBER = 150
logLMIN = np.log(ELLMAX_NOLIMBER)
logLMAX = np.log(60000+1)
NCell = 300
dlogL = (logLMAX - logLMIN)/(NCell - 1)

ells = np.zeros(ELLMAX_NOLIMBER+NCell)
for i in range(ELLMAX_NOLIMBER):
    ells[i] = i
for i in range(NCell):
    ells[i+ELLMAX_NOLIMBER] = np.exp(logLMIN + dlogL*i)
ells = ells[1:]

gammat_cl = []
wtheta_cl = []

for i in range(nlens):
    for j in range(nsrcs):
        gammat_cl.append(ccl.angular_cl(COSMO, lens[i], srcs[j], ells, l_limber=150) )
        
for i in range(nlens):
    wtheta_cl.append(ccl.angular_cl(COSMO, lens[i], lens[i], ells, l_limber=150))
    
#calculate the correlation function
tmin = 2.5
tmax = 250.
#ntheta = 15

logtmin = np.log(tmin)
logtmax = np.log(tmax)
logdt=(logtmax - logtmin)/ntheta
fac = (2./3.)
thetas = np.zeros(ntheta)

for i in range(ntheta):
    thetamin = np.exp(logtmin + (i + 0.)*logdt)
    thetamax = np.exp(logtmin + (i + 1.)*logdt)
    thetas[i] = fac * (thetamax**3 - thetamin**3) / (thetamax*thetamax    - thetamin*thetamin)
thetas /= 60

theta_b  = np.geomspace(tmin, tmax, ntheta+1) / 60
theta_u  = theta_b[1:]
theta_l  = theta_b[:-1]

ncombo_gammat = int(nsrcs*nlens)
ncombo_wtheta = int(nlens)

gammat, wtheta = [],[]

for i in range(ncombo_gammat):
    gammat.append(correlation_binned(COSMO, ell=ells, C_ell=gammat_cl[i], theta_min=theta_l, theta_max=theta_u, type='NG', method='Legendre'))
gammat = np.concatenate(gammat)

for i in range(ncombo_wtheta):
    wtheta.append(correlation_binned(COSMO, ell=ells, C_ell=wtheta_cl[i], theta_min=theta_l, theta_max=theta_u, type='NN', method='Legendre'))
wtheta = np.concatenate(wtheta)
 
nolimber_RSD_gammat = gammat
nolimber_RSD_wtheta = wtheta

In [8]:
print('!under NonLimber!')
chi2_gammat = (nolimber_RSD_gammat - nolimber_noRSD_gammat)@invcov_gammat@(nolimber_RSD_gammat - nolimber_noRSD_gammat)
print(f'The RSD raises chi2 {chi2_gammat:.2f} for gammat')
chi2_wtheta = (nolimber_RSD_wtheta - nolimber_noRSD_wtheta)@invcov_wtheta@(nolimber_RSD_wtheta - nolimber_noRSD_wtheta)
print(f'The RSD raises chi2 {chi2_wtheta:.2f} for wtheta')

!under NonLimber!
The RSD raises chi2 0.00 for gammat
The RSD raises chi2 7.64 for wtheta
